# Boomer → Gen Alpha Clean Dataset Generator — Local VS Code Version

This notebook is configured for local execution in VS Code.

Expected folder:

```text
/Users/dzakyrizha/Downloads/NLP_GenAlpha_Dataset_Generation/
├── input/
│   └── BoomerAlpha50K.csv
├── output/
├── partials/
├── logs/
└── .env
```

Put your Gemini API key in `.env`:

```env
GEMINI_API_KEY=your_api_key_here
```

Run on CPU. No GPU is required because generation uses the Gemini API.

In [1]:
# =============================================================================
# 2. IMPORTS + LOCAL PROJECT SETUP
# =============================================================================
from pathlib import Path
import os
import json
import time
import random

import pandas as pd
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv

# Local project folder
PROJECT_DIR = Path("/Users/dzakyrizha/Downloads/NLP_GenAlpha_Dataset_Generation")
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "output"
PARTIAL_DIR = PROJECT_DIR / "partials"
LOG_DIR = PROJECT_DIR / "logs"

for d in [INPUT_DIR, OUTPUT_DIR, PARTIAL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_CSV = INPUT_DIR / "BoomerAlpha50K.csv"

# Output paths for this 25K generation run
STRATIFIED_SOURCE_CSV = OUTPUT_DIR / "source_25000_stratified.csv"
FINAL_OUTPUT_CSV = OUTPUT_DIR / "generated_25000_gemini25_flashlite_prompt_final.csv"
PARTIAL_GENERATED_CSV = PARTIAL_DIR / "generated_25000_partial_prompt_final.csv"
ERROR_LOG = LOG_DIR / "generated_25000_errors_prompt_final.txt"
FAILED_BATCHES_CSV = LOG_DIR / "failed_generation_batches_25000_prompt_final.csv"

print("Project folder:", PROJECT_DIR)
print("Source exists:", SOURCE_CSV.exists())
print("Source path:", SOURCE_CSV)

if not SOURCE_CSV.exists():
    raise FileNotFoundError(f"Dataset not found at {SOURCE_CSV}. Put BoomerAlpha50K.csv inside the input folder.")

Project folder: /Users/dzakyrizha/Downloads/NLP_GenAlpha_Dataset_Generation
Source exists: True
Source path: /Users/dzakyrizha/Downloads/NLP_GenAlpha_Dataset_Generation/input/BoomerAlpha50K.csv


In [2]:
# =============================================================================
# 3. GEMINI API SETUP FROM .env
# =============================================================================
from google import genai

# Load .env from project folder
load_dotenv(PROJECT_DIR / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if GEMINI_API_KEY is None or not GEMINI_API_KEY.strip():
    raise ValueError(
        "GEMINI_API_KEY not found. Create a .env file in the project folder with: GEMINI_API_KEY=your_api_key_here"
    )

client = genai.Client(api_key=GEMINI_API_KEY)

# If this model is unavailable in your account, change it to the exact name from Google AI Studio.
GEMINI_MODEL = "gemini-2.5-flash-lite"

print("Gemini client ready.")
print("Model:", GEMINI_MODEL)

Gemini client ready.
Model: gemini-2.5-flash-lite


In [3]:
# =============================================================================
# 4. LOAD DATASET + CLEAN SOURCE DATA
# =============================================================================
df = pd.read_csv(SOURCE_CSV)

print("Original dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

if "boomer" not in df.columns:
    raise ValueError("Dataset must contain a 'boomer' column.")

metadata_cols = ["topic", "boomer_style", "gen_alpha_style", "sentence_type"]

for col in metadata_cols:
    if col not in df.columns:
        print(f"Missing metadata column {col!r}, filling with 'unknown'.")
        df[col] = "unknown"

# Keep only needed source-side columns
src_all = df[["boomer", "topic", "boomer_style", "gen_alpha_style", "sentence_type"]].copy()

# Basic cleaning
for col in src_all.columns:
    src_all[col] = src_all[col].fillna("unknown").astype(str).str.strip()
    src_all.loc[src_all[col] == "", col] = "unknown"

src_all = src_all[src_all["boomer"] != "unknown"].copy()

# Remove duplicate source rows
src_all = src_all.drop_duplicates(
    subset=["boomer", "topic", "boomer_style", "gen_alpha_style", "sentence_type"]
).reset_index(drop=True)

print("Clean source dataset shape:", src_all.shape)
display(src_all.head())

Original dataset shape: (49910, 6)
Columns: ['boomer', 'gen_alpha', 'topic', 'boomer_style', 'gen_alpha_style', 'sentence_type']
Clean source dataset shape: (49910, 5)


,boomer,topic,boomer_style,gen_alpha_style,sentence_type
0,"You should really get some more fresh air, it'...",health and medicine,unsolicited advice-giver,delulu and delusional optimism,exclamations or reactions
1,Make sure to take your vitamins every day.,health and medicine,unsolicited advice-giver,delulu and delusional optimism,exclamations or reactions
2,You need to eat more vegetables for better hea...,health and medicine,unsolicited advice-giver,delulu and delusional optimism,exclamations or reactions
3,Don't forget to hydrate throughout the day.,health and medicine,unsolicited advice-giver,delulu and delusional optimism,exclamations or reactions
4,You should really get your annual check-up.,health and medicine,unsolicited advice-giver,delulu and delusional optimism,exclamations or reactions


In [4]:
# =============================================================================
# 5. STRATIFIED SAMPLING SOURCE ROWS TO 25,000
# =============================================================================
TARGET_N = 25000
SEED = 42

# Stratify mainly by topic + sentence_type
src_all = src_all.copy()
src_all["strata"] = (
    src_all["topic"].astype(str) + " | " + src_all["sentence_type"].astype(str)
)

if len(src_all) <= TARGET_N:
    src_df = src_all.copy()
else:
    sampled_parts = []
    strata_counts = src_all["strata"].value_counts()
    total = len(src_all)

    for strata, count in strata_counts.items():
        group = src_all[src_all["strata"] == strata]

        # Proportional allocation
        n_sample = round((count / total) * TARGET_N)

        # Ensure at least 1 row from every stratum
        n_sample = max(1, n_sample)

        # Do not sample more than available
        n_sample = min(n_sample, len(group))

        sampled_parts.append(group.sample(n=n_sample, random_state=SEED))

    src_df = pd.concat(sampled_parts).drop_duplicates().reset_index(drop=True)

    # Adjust if over target
    if len(src_df) > TARGET_N:
        src_df = src_df.sample(n=TARGET_N, random_state=SEED).reset_index(drop=True)

    # Adjust if under target
    elif len(src_df) < TARGET_N:
        # More robust remaining selection using merge indicator
        base_cols = ["boomer", "topic", "boomer_style", "gen_alpha_style", "sentence_type"]
        current_keys = src_df[base_cols].copy()
        remaining = src_all.drop(columns=["strata"], errors="ignore").merge(
            current_keys,
            on=base_cols,
            how="left",
            indicator=True,
        )
        remaining = remaining[remaining["_merge"] == "left_only"].drop(columns=["_merge"])

        needed = TARGET_N - len(src_df)
        if len(remaining) >= needed:
            extra = remaining.sample(n=needed, random_state=SEED)
            src_df = pd.concat([src_df.drop(columns=["strata"], errors="ignore"), extra]).reset_index(drop=True)

# Remove helper stratum column
src_df = src_df.drop(columns=["strata"], errors="ignore").reset_index(drop=True)

# Assign stable row_id from 0 to N-1
src_df = src_df.reset_index().rename(columns={"index": "row_id"})

# Save stratified source rows
src_df.to_csv(STRATIFIED_SOURCE_CSV, index=False)

print("Final source rows to generate:", len(src_df))
print("Saved stratified source to:", STRATIFIED_SOURCE_CSV)

print("\nTopic coverage:", src_df["topic"].nunique())
print("Sentence type coverage:", src_df["sentence_type"].nunique())
print("Boomer style coverage:", src_df["boomer_style"].nunique())
print("Gen Alpha style coverage:", src_df["gen_alpha_style"].nunique())

print("\nTopic distribution:")
display(src_df["topic"].value_counts())

print("\nSentence type distribution:")
display(src_df["sentence_type"].value_counts())

print("\nTopic + sentence_type distribution:")
display(
    src_df.groupby(["topic", "sentence_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

Final source rows to generate: 25000
Saved stratified source to: /Users/dzakyrizha/Downloads/NLP_GenAlpha_Dataset_Generation/output/source_25000_stratified.csv

Topic coverage: 15
Sentence type coverage: 8
Boomer style coverage: 12
Gen Alpha style coverage: 12

Topic distribution:


topic
gaming and hobbies               2094
money and work                   1858
health and medicine              1810
weather and environment          1799
technology and social media      1758
family and home                  1755
food and cooking                 1724
relationships and dating         1644
politics and news                1591
fashion and style                1552
travel and going out             1531
school and studying              1481
pop culture and entertainment    1468
sports and fitness               1468
mental health and feelings       1467
Name: count, dtype: int64


Sentence type distribution:


sentence_type
compliments or praise         3504
complaints or frustrations    3246
requests or suggestions       3240
warnings or advice            3180
questions                     3119
commands or instructions      3016
exclamations or reactions     2849
observations or statements    2846
Name: count, dtype: int64


Topic + sentence_type distribution:


,topic,sentence_type,count
30,gaming and hobbies,requests or suggestions,389
31,gaming and hobbies,warnings or advice,373
53,money and work,questions,372
103,technology and social media,warnings or advice,363
34,health and medicine,compliments or praise,358
24,gaming and hobbies,commands or instructions,348
118,weather and environment,requests or suggestions,325
74,relationships and dating,compliments or praise,322
37,health and medicine,questions,320
58,politics and news,compliments or praise,317


In [5]:
# =============================================================================
# 6. FINAL REVISED PROMPT FUNCTION
# =============================================================================
def build_generation_prompt(batch_df):
    rows = []

    for _, row in batch_df.iterrows():
        rows.append({
            "row_id": int(row["row_id"]),
            "topic": str(row["topic"]),
            "boomer_style": str(row["boomer_style"]),
            "target_gen_alpha_style": str(row["gen_alpha_style"]),
            "sentence_type": str(row["sentence_type"]),
            "boomer": str(row["boomer"]),
        })

    prompt = f"""
You are creating a high-quality NLP dataset for Boomer-to-Gen Alpha STYLE TRANSLATION.

Task:
Rewrite each Boomer-style sentence into Gen Alpha-style language.

This is NOT a dialogue-response task.
The target sentence must NOT reply to, argue with, defend against, mock, reject, or contradict the source sentence.
The target sentence must preserve the original meaning, intention, and speech act.

Core objective:
Boomer-style sentence -> same meaning in Gen Alpha-style sentence

Critical rules:
1. Preserve the exact same meaning.
2. Preserve the same intention.
3. Preserve the same speech act.
4. If the source is a command, the target must remain a command.
5. If the source is advice, the target must remain advice.
6. If the source is a warning, the target must remain a warning.
7. If the source is praise, the target must remain praise.
8. If the source is encouragement, the target must remain encouragement.
9. Do not create an excuse, emotional reaction, comeback, or defensive response.
10. Only change the style, not the message.
11. Keep the output school-appropriate.
12. The target should sound natural, not forced or cringe.
13. Return one output for every input row. Do not skip any row.

Style rules:
1. Use clearly informal Gen Alpha / internet-native phrasing, but prioritize meaning preservation over slang intensity.
2. Do not overuse slang.
3. Avoid repeating the same slang too often.
4. Do not use \"vibe\", \"no cap\", \"main character\", \"legendary\", \"low-key\", \"like\", \"slay\", or \"fam\" too often.
5. Use varied phrasing across rows.
6. Prefer clear, casual, internet-native wording over random slang.
7. Avoid awkward metaphors.
8. The output should sound like something a real young person might actually say.
9. Mild slang is better than forced slang if stronger slang would change the meaning.
10. Do not make the sentence too dramatic unless the source is already dramatic.

Scoring rules:
- Use score 5 only when the target preserves the meaning and speech act almost perfectly.
- Use score 4 when the target mostly preserves meaning but has a small metaphorical shift, simplification, or tone adjustment.
- Do not assign 5 to every row.
- If a row is difficult, still produce the best meaning-preserving style translation and assign score 4.
- If the target becomes a reply, contradiction, excuse, or emotional reaction, rewrite it until it becomes a valid style translation.
- Only output semantic_preservation_score 4 or 5.

Bad examples:

Source: Stop playing games and clean your room.
Wrong target: Cleaning is for NPCs, I am the main character.
Reason: This contradicts the command.

Source: You should stop staying up late every night.
Wrong target: Late nights are when the magic happens, trust.
Reason: This gives the opposite advice.

Source: Spending too much money is irresponsible.
Wrong target: I buy what I want because life is short.
Reason: This is a defensive reply, not a translation.

Source: Put your phone down when someone is talking.
Wrong target: My phone understands me better than people.
Reason: This is an emotional response, not a translation.

Good examples:

Source: Stop playing games and clean your room.
Correct target: Pause the game rn and clean your room, side quest time.
Score: 5
Reason: The command is preserved and rewritten with casual internet style.

Source: You should stop staying up late every night.
Correct target: Staying up late every night is not the move, go get some sleep.
Score: 5
Reason: The advice is preserved and rewritten naturally.

Source: Spending too much money is irresponsible.
Correct target: Blowing all your money on random stuff is not it.
Score: 5
Reason: The warning is preserved without becoming defensive.

Source: Reading could help improve your mental well-being.
Correct target: Reading can be a solid brain reset when life feels heavy.
Score: 4
Reason: The advice is mostly preserved, but the wording slightly shifts the mental well-being idea into a metaphor.

Source: Proper posture is important for your back.
Correct target: Good posture helps your back stay unbothered.
Score: 5
Reason: The health advice is preserved with casual phrasing.

Source: You always volunteer to help at home.
Correct target: You always help out at home, and that is actually so clutch.
Score: 5
Reason: The praise is preserved using natural informal wording.

Source: You should be careful about which medicines you take.
Correct target: Be careful with the meds you take, do not just wing it.
Score: 5
Reason: The warning is preserved clearly with casual phrasing.

Return JSON only.
Return a list of objects with this exact schema:
[
  {{
    \"row_id\": 0,
    \"gen_alpha\": \"rewritten sentence\",
    \"semantic_preservation_score\": 5,
    \"notes\": \"short explanation\"
  }}
]

Important output requirements:
- Return valid JSON only.
- Return a JSON list, not a dictionary.
- Include every row_id from the input rows exactly once.
- Do not include markdown fences.
- Do not add extra commentary outside the JSON.

Rows:
{json.dumps(rows, ensure_ascii=False)}
"""
    return prompt

In [6]:
# =============================================================================
# 7. ROBUST JSON PARSING
# =============================================================================
def parse_json_response(text):
    if text is None:
        raise ValueError("Response text is None.")

    text = text.strip()

    # Remove markdown code fences if present
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 2:
            text = parts[1].strip()
            if text.startswith("json"):
                text = text[4:].strip()

    # Direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Extract JSON list
    start = text.find("[")
    end = text.rfind("]")

    if start != -1 and end != -1 and end > start:
        extracted = text[start:end + 1]
        return json.loads(extracted)

    raise ValueError("Could not parse JSON response.")

In [7]:
# =============================================================================
# 8. GENERATION SETTINGS + ROBUST GENERATE FUNCTION
# =============================================================================
BATCH_SIZE = 10
SLEEP_SECONDS = 3
MAX_RETRIES = 8
BASE_SLEEP_SECONDS = 5

def generate_batch(batch_df):
    prompt = build_generation_prompt(batch_df)

    expected_ids = set(batch_df["row_id"].astype(int).tolist())

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
            )

            text = response.text
            parsed = parse_json_response(text)

            if not isinstance(parsed, list):
                raise ValueError("Parsed response is not a list.")

            cleaned = []
            returned_ids = set()

            for item in parsed:
                if not isinstance(item, dict):
                    continue

                if "row_id" not in item:
                    continue

                row_id = int(item["row_id"])

                if row_id not in expected_ids:
                    continue

                gen_alpha = str(item.get("gen_alpha", "")).strip()
                score = item.get("semantic_preservation_score", 4)
                notes = str(item.get("notes", "")).strip()

                try:
                    score = int(float(score))
                except Exception:
                    score = 4

                # Clamp score to 4 or 5 only
                if score < 4:
                    score = 4
                if score > 5:
                    score = 5

                if not gen_alpha:
                    continue

                cleaned.append({
                    "row_id": row_id,
                    "gen_alpha": gen_alpha,
                    "semantic_preservation_score": score,
                    "notes": notes,
                })

                returned_ids.add(row_id)

            missing_ids = expected_ids - returned_ids
            if missing_ids:
                raise ValueError(f"Missing row_id(s) in response: {sorted(missing_ids)}")

            return cleaned

        except Exception as e:
            print(f"\nGeneration attempt {attempt}/{MAX_RETRIES} failed.")
            print("Error:", repr(e))

            with open(ERROR_LOG, "a", encoding="utf-8") as f:
                f.write(f"\nAttempt {attempt}/{MAX_RETRIES} failed.\n")
                f.write(f"Batch row_ids: {batch_df['row_id'].astype(int).tolist()}\n")
                f.write(f"Error: {repr(e)}\n")

            # Exponential backoff + jitter
            sleep_time = BASE_SLEEP_SECONDS * (2 ** (attempt - 1))
            sleep_time = min(sleep_time, 120)
            sleep_time += random.uniform(0, 3)

            print(f"Waiting {sleep_time:.1f} seconds before retrying...")
            time.sleep(sleep_time)

    return None

In [11]:
# =============================================================================
# 9. RUN GENERATION WITH AUTOSAVE + RESUME
# =============================================================================
generated_rows = []

# Resume existing partial if available
if PARTIAL_GENERATED_CSV.exists():
    partial_df = pd.read_csv(PARTIAL_GENERATED_CSV)

    if not partial_df.empty and "row_id" in partial_df.columns:
        partial_df["row_id"] = partial_df["row_id"].astype(int)
        partial_df = partial_df.drop_duplicates(subset=["row_id"], keep="last")
        generated_rows = partial_df.to_dict("records")
        done_ids = set(partial_df["row_id"].astype(int).tolist())
        print("Resuming from partial. Already generated:", len(done_ids))
    else:
        done_ids = set()
        print("Partial exists but invalid/empty. Starting fresh.")
else:
    done_ids = set()
    print("Starting fresh.")

pending_df = src_df[~src_df["row_id"].isin(done_ids)].copy()

print("Total target rows:", len(src_df))
print("Already done:", len(done_ids))
print("Pending rows:", len(pending_df))

failed_batches = []

# Load existing failed batches if available
if FAILED_BATCHES_CSV.exists():
    try:
        old_failed = pd.read_csv(FAILED_BATCHES_CSV)
        failed_batches = old_failed.to_dict("records")
    except Exception:
        failed_batches = []

for start in tqdm(range(0, len(pending_df), BATCH_SIZE)):
    batch_df = pending_df.iloc[start:start + BATCH_SIZE].copy()
    batch_ids = batch_df["row_id"].astype(int).tolist()

    outputs = generate_batch(batch_df)

    if outputs is not None:
        generated_rows.extend(outputs)

        temp_df = pd.DataFrame(generated_rows)

        if not temp_df.empty:
            temp_df["row_id"] = temp_df["row_id"].astype(int)
            temp_df = temp_df.drop_duplicates(subset=["row_id"], keep="last")
            temp_df = temp_df.sort_values("row_id").reset_index(drop=True)

            generated_rows = temp_df.to_dict("records")

            temp_df.to_csv(PARTIAL_GENERATED_CSV, index=False)
            print(f"Saved partial generated rows: {len(temp_df)}")

    else:
        print(f"Batch failed after all retries: {batch_ids}")

        failed_batches.append({
            "start_index": int(start),
            "row_ids": json.dumps(batch_ids),
            "status": "failed_after_retries",
        })

        pd.DataFrame(failed_batches).to_csv(FAILED_BATCHES_CSV, index=False)

    time.sleep(SLEEP_SECONDS)

generated_df = pd.DataFrame(generated_rows)

if not generated_df.empty:
    generated_df["row_id"] = generated_df["row_id"].astype(int)
    generated_df = generated_df.drop_duplicates(subset=["row_id"], keep="last")
    generated_df = generated_df.sort_values("row_id").reset_index(drop=True)

print("Generated rows:", len(generated_df))
display(generated_df.head())

if failed_batches:
    print("Some batches failed. Saved failed batch list to:")
    print(FAILED_BATCHES_CSV)
else:
    print("No failed batches.")

Resuming from partial. Already generated: 15380
Total target rows: 25000
Already done: 15380
Pending rows: 9620


  0%|          | 0/962 [00:00<?, ?it/s]

Saved partial generated rows: 15390


  0%|          | 1/962 [00:07<1:54:00,  7.12s/it]

Saved partial generated rows: 15400


...

Saved partial generated rows: 25000


100%|██████████| 962/962 [1:47:23<00:00,  6.70s/it]

Generated rows: 25000


,row_id,gen_alpha,semantic_preservation_score,notes
0,0,Setting goals is key when you're gaming compet...,5,The suggestion about setting goals in competit...
1,1,What's the appeal of playing those super spook...,4,The question about enjoying scary games is rep...
2,2,You should get outside more instead of staying...,5,The advice to engage in outdoor activities is ...
3,3,"Follow the game rules, or it's over.",4,The suggestion to respect game rules is transl...
4,4,Maybe step away from gaming for a bit so you d...,5,The advice to take a break from gaming to prev...


No failed batches.


In [12]:
# =============================================================================
# 10. CHECK COMPLETENESS
# =============================================================================
expected_ids = set(src_df["row_id"].astype(int).tolist())

if generated_df.empty:
    generated_ids = set()
else:
    generated_ids = set(generated_df["row_id"].astype(int).tolist())

missing_ids = sorted(expected_ids - generated_ids)

print("Expected rows:", len(expected_ids))
print("Generated rows:", len(generated_ids))
print("Missing rows:", len(missing_ids))

if missing_ids:
    print("First 50 missing row_ids:")
    print(missing_ids[:50])
    print("\nRerun the generation cell later. Resume logic will only process missing rows.")
else:
    print("All rows generated successfully.")

Expected rows: 25000
Generated rows: 25000
Missing rows: 0
All rows generated successfully.


In [13]:
# =============================================================================
# 11. MERGE SOURCE + GENERATED TARGETS
# =============================================================================
if generated_df.empty:
    raise ValueError("No generated rows found. Run generation first.")

translation_df = src_df.merge(generated_df, on="row_id", how="inner")

# Rename row_id to id
translation_df = translation_df.rename(columns={"row_id": "id"})

# Final column order
translation_df = translation_df[
    [
        "id",
        "topic",
        "boomer_style",
        "gen_alpha_style",
        "sentence_type",
        "boomer",
        "gen_alpha",
        "semantic_preservation_score",
        "notes",
    ]
].copy()

# Clean
for col in ["boomer", "gen_alpha", "topic", "boomer_style", "gen_alpha_style", "sentence_type", "notes"]:
    translation_df[col] = translation_df[col].fillna("").astype(str).str.strip()

translation_df["semantic_preservation_score"] = pd.to_numeric(
    translation_df["semantic_preservation_score"],
    errors="coerce"
).fillna(4).astype(int)

translation_df = translation_df[
    (translation_df["boomer"] != "") &
    (translation_df["gen_alpha"] != "")
].copy()

translation_df = translation_df.drop_duplicates(
    subset=["boomer", "gen_alpha"]
).reset_index(drop=True)

translation_df.to_csv(FINAL_OUTPUT_CSV, index=False)

print("Final generated dataset shape:", translation_df.shape)
print("Saved to:", FINAL_OUTPUT_CSV)

display(translation_df.head(20))

Final generated dataset shape: (25000, 9)
Saved to: /Users/dzakyrizha/Downloads/NLP_GenAlpha_Dataset_Generation/output/generated_25000_gemini25_flashlite_prompt_final.csv


,id,topic,boomer_style,gen_alpha_style,sentence_type,boomer,gen_alpha,semantic_preservation_score,notes
0,0,gaming and hobbies,full lecture mode,main character vibes,requests or suggestions,It’s important to set goals when gaming compet...,Setting goals is key when you're gaming compet...,5,The suggestion about setting goals in competit...
1,1,gaming and hobbies,confused by technology,delulu and delusional optimism,requests or suggestions,How can you enjoy playing those scary games?,What's the appeal of playing those super spook...,4,The question about enjoying scary games is rep...
2,2,gaming and hobbies,unsolicited advice-giver,NPC brainrot,requests or suggestions,You could benefit from some outdoor activities...,You should get outside more instead of staying...,5,The advice to engage in outdoor activities is ...
3,3,gaming and hobbies,full lecture mode,villain arc mentality,requests or suggestions,Respect the rules of each game you play.,"Follow the game rules, or it's over.",4,The suggestion to respect game rules is transl...
4,4,gaming and hobbies,unsolicited advice-giver,NPC brainrot,requests or suggestions,You might want to take a break from gaming bef...,Maybe step away from gaming for a bit so you d...,5,The advice to take a break from gaming to prev...
5,5,gaming and hobbies,overly literal and pedantic,deadpan sarcastic,requests or suggestions,It's important to have a backup plan aside fro...,Having a backup plan that isn't just gaming is...,5,The suggestion for an alternative plan to gami...
6,6,gaming and hobbies,full lecture mode,villain arc mentality,requests or suggestions,Practice regularly to improve your skills in y...,Grind consistently to level up your skills in ...,5,The advice on practicing regularly is translat...
7,7,gaming and hobbies,overly literal and pedantic,delulu and delusional optimism,requests or suggestions,I think you should take a break from gaming to...,Consider taking a gaming break to reset your b...,5,The suggestion to take a gaming break for ment...
8,8,gaming and hobbies,overly polite to the point of awkward,ironic hype,requests or suggestions,Perhaps you could take on a leadership role wi...,You could totally lead your gaming crew.,5,The suggestion to take a leadership role in a ...
9,9,gaming and hobbies,full lecture mode,delulu and delusional optimism,requests or suggestions,Dedicate time to learning about coding or game...,Spend time learning how to code or make games.,5,The suggestion to learn about coding or game d...


In [17]:
# =============================================================================
# 12. QUALITY CHECK SAMPLES
# =============================================================================
sample_n = min(30, len(translation_df))
sample_df = translation_df.sample(sample_n, random_state=42)

for _, row in sample_df.iterrows():
    print("=" * 100)
    print("ID:", row["id"])
    print("Topic:", row["topic"])
    print("Sentence type:", row["sentence_type"])
    print("Boomer style:", row["boomer_style"])
    print("Target style:", row["gen_alpha_style"])
    print("\nBoomer:")
    print(row["boomer"])
    print("\nGenerated Gen Alpha:")
    print(row["gen_alpha"])
    print("\nScore:", row["semantic_preservation_score"])
    print("Notes:", row["notes"])

ID: 6868
Topic: mental health and feelings
Sentence type: questions
Boomer style: overly polite to the point of awkward
Target style: extremely overdramatic

Boomer:
Would it help to take a moment to reflect on your thoughts?

Generated Gen Alpha:
Could taking a sec to think about your thoughts help?

Score: 5
Notes: Translates the polite suggestion to reflect into a more casual, question format.
ID: 24016
Topic: gaming and hobbies
Sentence type: compliments or praise
Boomer style: completely out of touch with youth culture
Target style: unhinged chaos energy

Boomer:
I admire your ability to focus on your interests.

Generated Gen Alpha:
It's cool seeing you lock in on the stuff you're into.

Score: 5
Notes: The original sentence praises focus on interests. The rewritten sentence uses casual, informal language to express admiration.
ID: 9668
Topic: family and home
Sentence type: requests or suggestions
Boomer style: worried and concerned parent
Target style: deadpan sarcastic

Boomer:

In [15]:
# =============================================================================
# 13. DISTRIBUTION CHECK
# =============================================================================
print("Final dataset rows:", len(translation_df))

print("\nTopic distribution:")
display(translation_df["topic"].value_counts())

print("\nSentence type distribution:")
display(translation_df["sentence_type"].value_counts())

print("\nScore distribution:")
display(translation_df["semantic_preservation_score"].value_counts())

print("\nTopic + sentence_type distribution:")
display(
    translation_df.groupby(["topic", "sentence_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(50)
)

Final dataset rows: 25000

Topic distribution:


topic
gaming and hobbies               2094
money and work                   1858
health and medicine              1810
weather and environment          1799
technology and social media      1758
family and home                  1755
food and cooking                 1724
relationships and dating         1644
politics and news                1591
fashion and style                1552
travel and going out             1531
school and studying              1481
pop culture and entertainment    1468
sports and fitness               1468
mental health and feelings       1467
Name: count, dtype: int64


Sentence type distribution:


sentence_type
compliments or praise         3504
complaints or frustrations    3246
requests or suggestions       3240
warnings or advice            3180
questions                     3119
commands or instructions      3016
exclamations or reactions     2849
observations or statements    2846
Name: count, dtype: int64


Score distribution:


semantic_preservation_score
5    19514
4     5486
Name: count, dtype: int64


Topic + sentence_type distribution:


,topic,sentence_type,count
30,gaming and hobbies,requests or suggestions,389
31,gaming and hobbies,warnings or advice,373
53,money and work,questions,372
103,technology and social media,warnings or advice,363
34,health and medicine,compliments or praise,358
24,gaming and hobbies,commands or instructions,348
118,weather and environment,requests or suggestions,325
74,relationships and dating,compliments or praise,322
37,health and medicine,questions,320
58,politics and news,compliments or praise,317


## Notes

If you get `503 UNAVAILABLE`, do not delete the partial CSV. Wait a few minutes and rerun cell 9. The notebook will resume from the missing rows only.

Final dataset path:

```text
/Users/dzakyrizha/Downloads/NLP_GenAlpha_Dataset_Generation/output/generated_25000_gemini25_flashlite_prompt_final.csv
```